In [128]:
import numpy as np
import pandas as pd
from typing import Dict

In [129]:
df = pd.read_csv("dataset/heart_failure_dataset.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [130]:
display(df.describe())
display(df.info())

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    str    
 2   ChestPainType   918 non-null    str    
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    str    
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    str    
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    str    
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 97.6 KB


None

In [131]:
label_column = "HeartDisease"

# Explicit ordinal relationships requested by user (low -> high)
ORDINAL_MAPPINGS = {
    "ChestPainType": ["ASY", "NAP", "ATA", "TA"],
    "ST_Slope": ["Up", "Flat", "Down"],
    "RestingECG": ["Normal", "ST", "LVH"],
}

all_categorical_columns = df.drop(columns=[label_column]).select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

ordinal_columns = [c for c in ORDINAL_MAPPINGS if c in df.columns]
nominal_category_columns = [
    c for c in all_categorical_columns if c not in ordinal_columns
]

numerical_columns = df.drop(columns=[label_column]).select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

feature_columns = df.drop(columns=[label_column]).columns.to_list()

display("Ordinal columns", ordinal_columns)
display("Nominal categorical columns", nominal_category_columns)
display("Numerical columns", numerical_columns)
display("feature_columns", feature_columns)


'Ordinal columns'

['ChestPainType', 'ST_Slope', 'RestingECG']

'Nominal categorical columns'

['Sex', 'ExerciseAngina']

'Numerical columns'

['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']

'feature_columns'

['Age',
 'Sex',
 'ChestPainType',
 'RestingBP',
 'Cholesterol',
 'FastingBS',
 'RestingECG',
 'MaxHR',
 'ExerciseAngina',
 'Oldpeak',
 'ST_Slope']

# Load & Preprocess Data

## Import

In [132]:
import yaml
from pathlib import Path
import pandas as pd


def _ensure_pandas_warning_compat():
    """Compatibility shim for pandas>=3 and libraries expecting SettingWithCopyWarning."""
    if not hasattr(pd.errors, "SettingWithCopyWarning"):
        class SettingWithCopyWarning(Warning):
            pass
        pd.errors.SettingWithCopyWarning = SettingWithCopyWarning

    # Some older call sites still look in pd.core.common
    try:
        if not hasattr(pd.core.common, "SettingWithCopyWarning"):
            pd.core.common.SettingWithCopyWarning = pd.errors.SettingWithCopyWarning
    except Exception:
        pass


_ensure_pandas_warning_compat()

import ray
from ray.data import Dataset

## Load configs

In [133]:
DEFAULT_CONFIG = {
    "data": {
        "dataset_path": "dataset/heart_failure_dataset.csv",
        "label_column": "HeartDisease",
        "test_size": 0.2,
        "split_seed": 42,
    },
    "encoding": {
        "ordinal_mappings": {
            "ChestPainType": ["ASY", "NAP", "ATA", "TA"],
            "ST_Slope": ["Up", "Flat", "Down"],
            "RestingECG": ["Normal", "ST", "LVH"],
        },
        "drop_first_baseline": True,
    },
    "preprocessing": {
        "scale_enabled": True,
        "process_full_dataset_single_batch": True,
        "map_batches_size": 2048,
    },
    "ray": {
        "auto_init": True,
        "ignore_reinit_error": True,
        "log_to_driver": False,
        "configure_logging": True,
        "logging_level": "CRITICAL",
        "data_verbose_progress": False,
    },
    "training": {
        "logistic": {
            "max_iter": 1000,
            "solver": "lbfgs",
            "class_weight": "balanced",
            "random_state": 42,
            "model_output_path": "./results/logistic_regression/model.joblib",
        },
        "xgboost": {
            "num_workers": 1,
            "use_gpu": False,
            "resources_per_worker": None,
            "max_boost_rounds": 500,
            "num_samples": 10,
            "max_depth_range": [1, 9],
            "min_child_weight_values": [1, 2, 3],
            "subsample_range": [0.5, 1.0],
            "eta_log_range": [0.0001, 0.1],
            "num_boost_round_values": [100, 200, 500],
            "objective": "binary:logistic",
            "eval_metric": ["logloss", "error"],
            "metric_name": "eval-logloss",
            "metric_mode": "min",
            "scheduler_grace_period": 10,
            "scheduler_reduction_factor": 3,
            "run_name": "HeartDisease_prediction_xgboost",
            "storage_path": "./results",
            "checkpoints_to_keep": 3,
        },
    },
    "evaluation": {
        "logistic_threshold": 0.5,
        "xgboost_threshold": 0.5,
        "xgboost_top_n": 3,
        "xgboost_model_filename": "model.ubj",
    },
}

def _deep_merge(base: dict, override: dict) -> dict:
    merged = dict(base)
    for k, v in (override or {}).items():
        if isinstance(v, dict) and isinstance(merged.get(k), dict):
            merged[k] = _deep_merge(merged[k], v)
        else:
            merged[k] = v
    return merged

def load_config(config_path: str = "config.yaml") -> Dict:
    """Load YAML config file and merge with defaults."""
    config_path_obj = Path(config_path)

    if not config_path_obj.exists():
        raise FileNotFoundError(f"config file not found : {config_path_obj}")

    with open(config_path_obj, "r", encoding="utf-8") as file:
        user_config = yaml.safe_load(file) or {}

    config = _deep_merge(DEFAULT_CONFIG, user_config)
    return config

## Load data

In [134]:
def load_data(path: str):
    """Load Data"""
    # TODO(config): data.dataset_path
    ds = ray.data.read_csv(path)
    # print(f"Schema: {ds.schema()}")
    # print(f"Loaded dataset with {ds.count()} rows")
    return ds

## Preprocess data 

In [135]:
from typing import Dict
import logging
import ray
from ray.data.preprocessors import Chain, OrdinalEncoder, OneHotEncoder, StandardScaler


def _ensure_pandas_warning_compat_worker() -> None:
    """Ensure pandas warning symbols exist in worker process context too."""
    import pandas as _pd

    if not hasattr(_pd.errors, "SettingWithCopyWarning"):
        class SettingWithCopyWarning(Warning):
            pass
        _pd.errors.SettingWithCopyWarning = SettingWithCopyWarning

    try:
        if not hasattr(_pd.core.common, "SettingWithCopyWarning"):
            _pd.core.common.SettingWithCopyWarning = _pd.errors.SettingWithCopyWarning
    except Exception:
        pass


def _worker_compat_batch(batch):
    _ensure_pandas_warning_compat_worker()
    return batch


def preprocess_data(ds: Dataset, cfg: Dict):
    """Preprocess with Ray-native preprocessors before train/test split."""
    ray_cfg = cfg.get("ray", {})
    pp_cfg = cfg.get("preprocessing", {})
    data_cfg = cfg.get("data", {})
    encoding_cfg = cfg.get("encoding", {})

    if ray_cfg.get("auto_init", True) and not ray.is_initialized():
        log_level_name = str(ray_cfg.get("logging_level", "ERROR")).upper()
        log_level = getattr(logging, log_level_name, logging.ERROR)
        ray.init(
            ignore_reinit_error=ray_cfg.get("ignore_reinit_error", True),
            log_to_driver=ray_cfg.get("log_to_driver", False),
            configure_logging=ray_cfg.get("configure_logging", True),
            logging_level=log_level,
        )

    # Keep Ray Data progress verbosity off by default for cleaner notebook output
    try:
        from ray.data import DataContext
        DataContext.get_current().execution_options.verbose_progress = bool(
            ray_cfg.get("data_verbose_progress", False)
        )
    except Exception:
        pass

    # Ensure worker-side pandas warning compatibility before preprocessors execute.
    ds = ds.map_batches(_worker_compat_batch, batch_format="numpy")

    # Keep behavior where the full dataset is transformed before split.
    ordinal_cols = [c for c in ORDINAL_MAPPINGS if c in ds.columns()]
    nominal_cols = [c for c in nominal_category_columns if c in ds.columns()]

    # Scale numeric + ordinal-encoded columns after encoders are applied.
    numeric_plus_ordinal = [
        c for c in (numerical_columns + ordinal_cols) if c in ds.columns()
    ]

    preprocessors = []
    if ordinal_cols:
        preprocessors.append(OrdinalEncoder(columns=ordinal_cols))

    if nominal_cols:
        preprocessors.append(OneHotEncoder(columns=nominal_cols))

    if pp_cfg.get("scale_enabled", True) and numeric_plus_ordinal:
        preprocessors.append(StandardScaler(columns=numeric_plus_ordinal))

    ray_preprocessor = Chain(*preprocessors) if preprocessors else None

    if ray_preprocessor is not None:
        processed_ds = ray_preprocessor.fit_transform(ds)
    else:
        processed_ds = ds

    test_size = data_cfg.get("test_size", 0.2)
    split_seed = data_cfg.get("split_seed", 42)
    train_ds, test_ds = processed_ds.train_test_split(test_size=test_size, seed=split_seed)

    preprocess_artifacts = {
        "ordinal_mappings": ORDINAL_MAPPINGS,
        "nominal_columns_encoded": nominal_cols,
        "drop_first_baseline": encoding_cfg.get("drop_first_baseline", True),
        "scaled_columns": numeric_plus_ordinal if pp_cfg.get("scale_enabled", True) else [],
        "ray_preprocessor": ray_preprocessor,
        "test_size": test_size,
        "split_seed": split_seed,
    }

    return train_ds, test_ds, preprocess_artifacts

# Train & Evaluate Data

## Train XGBoost model

In [136]:
import os
import xgboost
import ray
import ray.train
import pandas as pd
import numpy as np
from ray.train import ScalingConfig
from ray.data import Dataset
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from typing import Dict


def _prepare_xgb_matrix(
    df: pd.DataFrame,
    label_column: str,
    reference_columns: list | None = None,
) -> tuple[np.ndarray, np.ndarray | None, list[str]]:
    """Convert a dataframe into a strictly numeric 2D matrix for XGBoost.

    Handles sequence/object cells by expanding them into scalar columns.
    """
    work_df = df.copy()
    y = None
    if label_column in work_df.columns:
        y = pd.to_numeric(work_df[label_column], errors="coerce").fillna(0).to_numpy(dtype=np.int64)
        work_df = work_df.drop(columns=[label_column])

    cols_to_expand = []
    for col in work_df.columns:
        series = work_df[col]
        first_valid = None
        for val in series:
            if val is not None and not (isinstance(val, float) and np.isnan(val)):
                first_valid = val
                break
        if isinstance(first_valid, (list, tuple, np.ndarray)):
            cols_to_expand.append(col)

    for col in cols_to_expand:
        arr_series = work_df[col].apply(
            lambda x: np.asarray(x, dtype=np.float32).ravel()
            if isinstance(x, (list, tuple, np.ndarray))
            else np.asarray([x], dtype=np.float32) if pd.notna(x) else np.asarray([np.nan], dtype=np.float32)
        )
        max_len = int(arr_series.map(len).max())
        for i in range(max_len):
            work_df[f"{col}_{i}"] = arr_series.map(lambda arr: float(arr[i]) if i < len(arr) else np.nan)
        work_df = work_df.drop(columns=[col])

    work_df = work_df.apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)

    if reference_columns is not None:
        for c in reference_columns:
            if c not in work_df.columns:
                work_df[c] = 0.0
        work_df = work_df[reference_columns]

    feature_names = work_df.columns.tolist()
    X = work_df.to_numpy(dtype=np.float32, copy=False)
    return X, y, feature_names



def train_xgboost_model(
    train_ds: Dataset, test_ds: Dataset, label_column: str, cfg: Dict
):
    """Train XGBoost model with Ray Tune hyperparameter search.

    Optimization target is recall-priority with precision awareness using F-beta
    for label=1.
    """

    xgb_cfg = cfg.get("training", {}).get("xgboost", {})

    max_boost_rounds = int(xgb_cfg.get("max_boost_rounds", 500))
    num_samples = int(xgb_cfg.get("num_samples", 12))
    max_concurrent_trials = int(xgb_cfg.get("max_concurrent_trials", 2))

    # Tighter, quality-oriented priors
    max_depth_low, max_depth_high = xgb_cfg.get("max_depth_range", [3, 8])
    subsample_low, subsample_high = xgb_cfg.get("subsample_range", [0.7, 1.0])
    eta_low, eta_high = xgb_cfg.get("eta_log_range", [3e-3, 8e-2])

    decision_threshold = float(xgb_cfg.get("decision_threshold", 0.55))
    fbeta_beta = float(xgb_cfg.get("fbeta_beta", 1.2))
    beta_tag = str(fbeta_beta).replace(".", "")
    default_metric_name = f"validation-fbeta{beta_tag}"

    metric_name = str(xgb_cfg.get("metric_name", default_metric_name))
    metric_mode = xgb_cfg.get("metric_mode", "max")

    # Materialize once here, outside Tune trials
    train_df = train_ds.to_pandas()
    eval_df = test_ds.to_pandas()

    X_train, y_train, feature_names = _prepare_xgb_matrix(train_df, label_column)
    X_eval, y_eval, _ = _prepare_xgb_matrix(eval_df, label_column, reference_columns=feature_names)

    def train_fn(config: Dict):
        """Tune trainable function."""
        from sklearn.metrics import precision_score, recall_score, fbeta_score

        trial_id = tune.get_context().get_trial_id()
        dtrain = xgboost.DMatrix(X_train, label=y_train)
        deval = xgboost.DMatrix(X_eval, label=y_eval)

        params = {
            "objective": config.get("objective", "binary:logistic"),
            "eval_metric": config.get("eval_metric", ["logloss", "error"]),
            "max_depth": config["max_depth"],
            "min_child_weight": config["min_child_weight"],
            "subsample": config["subsample"],
            "colsample_bytree": config["colsample_bytree"],
            "eta": config["eta"],
            "gamma": config["gamma"],
            "reg_lambda": config["reg_lambda"],
            "reg_alpha": config["reg_alpha"],
            "scale_pos_weight": config["scale_pos_weight"],
            "tree_method": "hist",
        }

        evals_result = {}
        booster = xgboost.train(
            params,
            dtrain=dtrain,
            evals=[(deval, "validation")],
            num_boost_round=config["num_boost_round"],
            evals_result=evals_result,
            verbose_eval=False,
        )

        checkpoint_dir = os.path.abspath(
            os.path.join(
                xgb_cfg.get("storage_path", "./results"),
                xgb_cfg.get("run_name", "HeartDisease_prediction_xgboost"),
                f"trial_{trial_id}",
            )
        )
        os.makedirs(checkpoint_dir, exist_ok=True)

        booster.save_model(os.path.join(checkpoint_dir, "model.ubj"))
        checkpoint = ray.train.Checkpoint.from_directory(checkpoint_dir)

        eval_proba = booster.predict(deval)
        eval_pred = (eval_proba >= decision_threshold).astype(int)

        validation_precision = precision_score(y_eval, eval_pred, pos_label=1, zero_division=0)
        validation_recall = recall_score(y_eval, eval_pred, pos_label=1, zero_division=0)
        validation_fbeta = fbeta_score(
            y_eval,
            eval_pred,
            beta=fbeta_beta,
            pos_label=1,
            zero_division=0,
        )
        final_logloss = evals_result["validation"]["logloss"][-1]

        metrics_payload = {
            "validation-logloss": final_logloss,
            "validation-precision": validation_precision,
            "validation-recall": validation_recall,
            default_metric_name: validation_fbeta,
            "validation-fbeta12": validation_fbeta,
            "validation-fbeta15": validation_fbeta,
        }
        metrics_payload[metric_name] = validation_fbeta

        tune.report(metrics_payload, checkpoint=checkpoint)

    search_space = {
        "objective": xgb_cfg.get("objective", "binary:logistic"),
        "eval_metric": xgb_cfg.get("eval_metric", ["logloss", "error"]),
        "max_depth": tune.randint(int(max_depth_low), int(max_depth_high)),
        "min_child_weight": tune.choice(
            xgb_cfg.get("min_child_weight_values", [2, 4, 6, 8])
        ),
        "subsample": tune.uniform(float(subsample_low), float(subsample_high)),
        "colsample_bytree": tune.choice(
            xgb_cfg.get("colsample_bytree_values", [0.7, 0.85, 1.0])
        ),
        "eta": tune.loguniform(float(eta_low), float(eta_high)),
        "gamma": tune.choice(xgb_cfg.get("gamma_values", [0.0, 0.5, 1.0])),
        "reg_lambda": tune.choice(xgb_cfg.get("reg_lambda_values", [1.0, 2.0, 5.0, 10.0])),
        "reg_alpha": tune.choice(xgb_cfg.get("reg_alpha_values", [0.0, 0.1, 0.5])),
        "scale_pos_weight": tune.choice(
            xgb_cfg.get("scale_pos_weight_values", [1.0, 1.2, 1.4, 1.6])
        ),
        "num_boost_round": tune.choice(
            xgb_cfg.get("num_boost_round_values", [200, 400, 600])
        ),
    }

    scheduler = ASHAScheduler(
        max_t=max_boost_rounds,
        grace_period=int(xgb_cfg.get("scheduler_grace_period", 10)),
        reduction_factor=int(xgb_cfg.get("scheduler_reduction_factor", 3)),
    )

    raw_storage_path = xgb_cfg.get("storage_path", "./results")
    storage_path = (
        raw_storage_path
        if os.path.isabs(raw_storage_path) or "://" in raw_storage_path
        else os.path.abspath(raw_storage_path)
    )

    tuner = tune.Tuner(
        train_fn,
        param_space=search_space,
        tune_config=tune.TuneConfig(
            metric=metric_name,
            mode=metric_mode,
            scheduler=scheduler,
            num_samples=num_samples,
            max_concurrent_trials=max_concurrent_trials,
        ),
        run_config=tune.RunConfig(
            name=xgb_cfg.get("run_name", "HeartDisease_prediction_xgboost"),
            storage_path=storage_path,
            checkpoint_config=tune.CheckpointConfig(
                num_to_keep=int(xgb_cfg.get("checkpoints_to_keep", 3)),
                checkpoint_score_attribute=metric_name,
                checkpoint_score_order=metric_mode,
            ),
        ),
    )

    return tuner.fit()

## Evaluate XGBoost model

In [137]:
import os
import numpy as np
import xgboost
from typing import Dict, List, Optional
from ray.tune import ResultGrid
from ray.data import Dataset
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)


def evaluate_xgboost_model(
    result: ResultGrid,
    test_ds: Dataset,
    label_column: str,
    cfg: Dict,
    top_n: Optional[int] = None,
    threshold: Optional[float] = None,
) -> List[Dict]:
    """Evaluate top N tuned XGBoost models on held-out test set (config-driven)."""

    eval_cfg = cfg.get("evaluation", {})
    xgb_cfg = cfg.get("training", {}).get("xgboost", {})
    metric_name = xgb_cfg.get("metric_name", "validation-fbeta15")
    model_filename = eval_cfg.get("xgboost_model_filename", "model.ubj")

    if top_n is None:
        top_n = int(eval_cfg.get("xgboost_top_n", 3))
    if threshold is None:
        threshold = float(eval_cfg.get("xgboost_threshold", 0.5))

    all_results = sorted(
        [r for r in result if not r.error],
        key=lambda r: r.metrics[metric_name],
        reverse=True,
    )[:top_n]

    def evaluate_single(single_result, rank: int) -> Dict:
        checkpoint = single_result.checkpoint

        if checkpoint is None:
            raise ValueError(f"Trial rank {rank} has no checkpoint saved.")

        with checkpoint.as_directory() as checkpoint_dir:
            model = xgboost.Booster()
            model.load_model(os.path.join(checkpoint_dir, model_filename))

        test_df = test_ds.to_pandas()
        X_test, y_true, _ = _prepare_xgb_matrix(test_df, label_column)

        dmatrix = xgboost.DMatrix(X_test)
        y_proba = model.predict(dmatrix)
        y_pred = (y_proba > threshold).astype(int)

        tn, fp, _, _ = confusion_matrix(y_true, y_pred).ravel()

        print(classification_report(y_true, y_pred, zero_division=0))
        return {
            "rank": rank,
            "threshold": threshold,
            "tune_metric_name": metric_name,
            "tune_metric_value": single_result.metrics[metric_name],
            "hyperparameters": single_result.config,
            "accuracy": accuracy_score(y_true, y_pred),
            "roc_auc": roc_auc_score(y_true, y_proba),
            "f1_label_1": f1_score(
                y_true, y_pred, pos_label=1, average="binary", zero_division=0
            ),
            "precision_label_1": precision_score(
                y_true, y_pred, pos_label=1, average="binary", zero_division=0
            ),
            "recall_label_1": recall_score(
                y_true, y_pred, pos_label=1, average="binary", zero_division=0
            ),
            "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
            "confusion_matrix": confusion_matrix(y_true, y_pred),
        }

    return [evaluate_single(r, rank=i + 1) for i, r in enumerate(all_results)]

# Pipeline

In [138]:
# XGBoost-only pipeline with config-driven preprocessing/training/evaluation
import logging

try:
    cfg = load_config("config.yaml")

    # TODO(config): map core schema values into notebook globals
    label_column = cfg.get("data", {}).get("label_column", label_column)
    ORDINAL_MAPPINGS = cfg.get("encoding", {}).get("ordinal_mappings", ORDINAL_MAPPINGS)

    # TODO(config): data.dataset_path
    dataset_path = cfg.get("data", {}).get("dataset_path", "dataset/heart_failure_dataset.csv")

    ray_cfg = cfg.get("ray", {})
    # Force Ray-related Python loggers to stay quiet in notebook output
    for logger_name in ["ray", "ray.data", "ray.train", "ray.tune"]:
        logger = logging.getLogger(logger_name)
        logger.setLevel(logging.CRITICAL)
        logger.propagate = False

    if not ray.is_initialized():
        log_level_name = str(ray_cfg.get("logging_level", "CRITICAL")).upper()
        log_level = getattr(logging, log_level_name, logging.CRITICAL)
        ray.init(
            ignore_reinit_error=ray_cfg.get("ignore_reinit_error", True),
            log_to_driver=ray_cfg.get("log_to_driver", False),
            configure_logging=ray_cfg.get("configure_logging", True),
            logging_level=log_level,
            object_store_memory=2 * 1024**3,
        )

    # Keep Ray Data progress verbosity off by default for cleaner notebook output
    try:
        from ray.data import DataContext

        DataContext.get_current().execution_options.verbose_progress = bool(
            ray_cfg.get("data_verbose_progress", False)
        )
    except Exception:
        pass

    ds = load_data(dataset_path)

    # Recompute column groups after config mapping
    profiling_df = ds.to_pandas()
    all_categorical_columns = profiling_df.drop(columns=[label_column]).select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    ordinal_columns = [c for c in ORDINAL_MAPPINGS if c in profiling_df.columns]
    nominal_category_columns = [c for c in all_categorical_columns if c not in ordinal_columns]
    numerical_columns = profiling_df.drop(columns=[label_column]).select_dtypes(
        include=["int64", "float64", "int32", "float32"]
    ).columns.tolist()

    train_ds, test_ds, preprocessor = preprocess_data(ds, cfg)
    print("Preprocessed Data Ready")

    xgb_results = train_xgboost_model(
        train_ds=train_ds,
        test_ds=test_ds,
        label_column=label_column,
        cfg=cfg,
    )

    print("Trained XGBoost Model")

    xgb_metrics = evaluate_xgboost_model(
        result=xgb_results,
        test_ds=test_ds,
        label_column=label_column,
        cfg=cfg,
    )

    print("\nXGBoost metrics (top models):")
    print(xgb_metrics)

except Exception as exc:
    print("***Error***", exc)

finally:
    if ray.is_initialized():
        ray.shutdown()

KeyboardInterrupt: 

In [ ]:
# Trial error diagnostics
if 'xgb_results' in globals():
    errored = [r for r in xgb_results if r.error]
    print('Errored trials:', len(errored), 'of', len(list(xgb_results)))
    if errored:
        print('First error:')
        print(errored[0].error)
else:
    print('xgb_results not available')

Errored trials: 0 of 16


In [ ]:
# Post-run sanity check
if 'xgb_results' in globals():
    errored = [r for r in xgb_results if r.error]
    ok = [r for r in xgb_results if not r.error]
    print('Trials ok:', len(ok), 'errored:', len(errored))
    if errored:
        print('First trial error:')
        print(errored[0].error)
else:
    print('xgb_results not found')

if 'xgb_metrics' in globals():
    print('xgb_metrics length:', len(xgb_metrics))
    if len(xgb_metrics):
        top = xgb_metrics[0]
        print('Top metric keys:', list(top.keys()))
        print('Top-1 summary:', {
            'tune_metric_name': top.get('tune_metric_name'),
            'tune_metric_value': round(float(top.get('tune_metric_value', 0.0)), 4),
            'accuracy': round(float(top.get('accuracy', 0.0)), 4),
            'precision_label_1': round(float(top.get('precision_label_1', 0.0)), 4),
            'recall_label_1': round(float(top.get('recall_label_1', 0.0)), 4),
            'f1_label_1': round(float(top.get('f1_label_1', 0.0)), 4),
            'roc_auc': round(float(top.get('roc_auc', 0.0)), 4),
        })

Trials ok: 16 errored: 0
xgb_metrics length: 3
Top metric keys: ['rank', 'threshold', 'tune_metric_name', 'tune_metric_value', 'hyperparameters', 'accuracy', 'roc_auc', 'f1_label_1', 'precision_label_1', 'recall_label_1', 'specificity', 'confusion_matrix']
Top-1 summary: {'tune_metric_name': 'validation-fbeta12', 'tune_metric_value': 0.7877, 'accuracy': 0.7391, 'precision_label_1': 0.6863, 'recall_label_1': 0.814, 'f1_label_1': 0.7447, 'roc_auc': 0.8767}


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, fbeta_score, accuracy_score, roc_auc_score, confusion_matrix, classification_report


def train_xgboost_baseline_cv(train_ds: Dataset, label_column: str, cfg: Dict):
    """Train a non-tuned XGBoost baseline using CV to pick num_boost_round."""
    xgb_cfg = cfg.get("training", {}).get("xgboost", {})
    baseline_cfg = cfg.get("training", {}).get("xgboost_baseline", {})

    train_df = train_ds.to_pandas()
    X, y, feature_names = _prepare_xgb_matrix(train_df, label_column)

    beta = float(baseline_cfg.get("fbeta_beta", xgb_cfg.get("fbeta_beta", 1.2)))
    threshold = float(baseline_cfg.get("decision_threshold", cfg.get("evaluation", {}).get("xgboost_threshold", 0.5)))
    cv_folds = int(baseline_cfg.get("cv_folds", 5))
    seed = int(cfg.get("data", {}).get("split_seed", 42))

    max_depth_range = xgb_cfg.get("max_depth_range", [3, 8])
    num_round_candidates = baseline_cfg.get("num_boost_round_candidates", [100, 200, 400, 600])

    # Fixed baseline params (no hyperparameter tuning)
    params = {
        "objective": xgb_cfg.get("objective", "binary:logistic"),
        "eval_metric": ["logloss", "error"],
        "max_depth": int(baseline_cfg.get("max_depth", int((max_depth_range[0] + max_depth_range[1]) / 2))),
        "min_child_weight": int(baseline_cfg.get("min_child_weight", 4)),
        "subsample": float(baseline_cfg.get("subsample", 0.9)),
        "colsample_bytree": float(baseline_cfg.get("colsample_bytree", 0.85)),
        "eta": float(baseline_cfg.get("eta", 0.03)),
        "gamma": float(baseline_cfg.get("gamma", 0.5)),
        "reg_lambda": float(baseline_cfg.get("reg_lambda", 2.0)),
        "reg_alpha": float(baseline_cfg.get("reg_alpha", 0.1)),
        "scale_pos_weight": float(baseline_cfg.get("scale_pos_weight", 1.2)),
        "tree_method": "hist",
    }

    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed)

    cv_rows = []
    best_round = None
    best_fbeta = -1.0

    for n_rounds in num_round_candidates:
        fold_precisions, fold_recalls, fold_fbetas = [], [], []

        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]

            dtr = xgboost.DMatrix(X_tr, label=y_tr)
            dval = xgboost.DMatrix(X_val, label=y_val)

            booster = xgboost.train(
                params,
                dtr,
                num_boost_round=int(n_rounds),
                evals=[(dval, "validation")],
                verbose_eval=False,
            )

            proba = booster.predict(dval)
            pred = (proba >= threshold).astype(int)

            p = precision_score(y_val, pred, pos_label=1, average="binary", zero_division=0)
            r = recall_score(y_val, pred, pos_label=1, average="binary", zero_division=0)
            f = fbeta_score(y_val, pred, beta=beta, pos_label=1, average="binary", zero_division=0)

            fold_precisions.append(p)
            fold_recalls.append(r)
            fold_fbetas.append(f)

        row = {
            "num_boost_round": int(n_rounds),
            "cv_precision_label_1_mean": float(np.mean(fold_precisions)),
            "cv_recall_label_1_mean": float(np.mean(fold_recalls)),
            "cv_fbeta_label_1_mean": float(np.mean(fold_fbetas)),
        }
        cv_rows.append(row)

        if row["cv_fbeta_label_1_mean"] > best_fbeta:
            best_fbeta = row["cv_fbeta_label_1_mean"]
            best_round = int(n_rounds)

    dtrain_full = xgboost.DMatrix(X, label=y)
    baseline_model = xgboost.train(params, dtrain_full, num_boost_round=best_round, verbose_eval=False)

    return {
        "model": baseline_model,
        "feature_names": feature_names,
        "best_num_boost_round": best_round,
        "best_cv_fbeta": best_fbeta,
        "cv_results": cv_rows,
        "threshold": threshold,
        "beta": beta,
        "params": params,
    }


def evaluate_xgboost_baseline_model(
    baseline_bundle: Dict,
    test_ds: Dataset,
    label_column: str,
):
    """Evaluate baseline XGBoost model on held-out test set."""
    model = baseline_bundle["model"]
    threshold = baseline_bundle["threshold"]

    test_df = test_ds.to_pandas()
    X_test, y_true, _ = _prepare_xgb_matrix(test_df, label_column, reference_columns=baseline_bundle["feature_names"])

    dtest = xgboost.DMatrix(X_test)
    y_proba = model.predict(dtest)
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, _, _ = confusion_matrix(y_true, y_pred).ravel()

    print(classification_report(y_true, y_pred, zero_division=0))
    return {
        "model": "xgboost_baseline_cv",
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "f1_label_1": fbeta_score(
            y_true, y_pred, beta=1.0, pos_label=1, average="binary", zero_division=0
        ),
        "precision_label_1": precision_score(
            y_true, y_pred, pos_label=1, average="binary", zero_division=0
        ),
        "recall_label_1": recall_score(
            y_true, y_pred, pos_label=1, average="binary", zero_division=0
        ),
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }

In [ ]:
# Baseline (CV, no tuning) training + evaluation + comparison with tuned top-1
import logging

# Recreate context if needed (e.g., after Ray was shutdown by the main pipeline cell)
if "cfg" not in globals():
    cfg = load_config("config.yaml")

ray_cfg = cfg.get("ray", {})
if not ray.is_initialized():
    log_level_name = str(ray_cfg.get("logging_level", "CRITICAL")).upper()
    log_level = getattr(logging, log_level_name, logging.CRITICAL)
    ray.init(
        ignore_reinit_error=ray_cfg.get("ignore_reinit_error", True),
        log_to_driver=ray_cfg.get("log_to_driver", False),
        configure_logging=ray_cfg.get("configure_logging", True),
        logging_level=log_level,
    )

# Always rebuild fresh train/test datasets in the current Ray session
dataset_path = cfg.get("data", {}).get("dataset_path", "dataset/heart_failure_dataset.csv")
label_column = cfg.get("data", {}).get("label_column", "HeartDisease")
baseline_ds = load_data(dataset_path)

profiling_df = baseline_ds.to_pandas()
_ordinal_mappings = cfg.get("encoding", {}).get("ordinal_mappings", ORDINAL_MAPPINGS)
all_categorical_columns = profiling_df.drop(columns=[label_column]).select_dtypes(
    include=["object", "category", "string"]
 ).columns.tolist()
ordinal_columns = [c for c in _ordinal_mappings if c in profiling_df.columns]
nominal_category_columns = [c for c in all_categorical_columns if c not in ordinal_columns]
numerical_columns = profiling_df.drop(columns=[label_column]).select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

baseline_train_ds, baseline_test_ds, _ = preprocess_data(baseline_ds, cfg)

baseline_bundle = train_xgboost_baseline_cv(train_ds=baseline_train_ds, label_column=label_column, cfg=cfg)
print("Baseline best num_boost_round:", baseline_bundle["best_num_boost_round"])
print("Baseline best CV F-beta:", round(baseline_bundle["best_cv_fbeta"], 4))
display(pd.DataFrame(baseline_bundle["cv_results"]))

baseline_metrics = evaluate_xgboost_baseline_model(
    baseline_bundle=baseline_bundle,
    test_ds=baseline_test_ds,
    label_column=label_column,
)
print("Baseline metrics:", baseline_metrics)

if "xgb_metrics" in globals() and len(xgb_metrics) > 0:
    tuned_top = xgb_metrics[0]
    comparison_df = pd.DataFrame([
        {
            "model": "baseline_cv",
            "accuracy": baseline_metrics["accuracy"],
            "precision_label_1": baseline_metrics["precision_label_1"],
            "recall_label_1": baseline_metrics["recall_label_1"],
            "f1_label_1": baseline_metrics["f1_label_1"],
            "roc_auc": baseline_metrics["roc_auc"],
            "specificity": baseline_metrics["specificity"],
        },
        {
            "model": "tuned_top1",
            "accuracy": tuned_top["accuracy"],
            "precision_label_1": tuned_top["precision_label_1"],
            "recall_label_1": tuned_top["recall_label_1"],
            "f1_label_1": tuned_top["f1_label_1"],
            "roc_auc": tuned_top["roc_auc"],
            "specificity": tuned_top["specificity"],
        },
    ])
    print("\nBaseline vs Tuned Top-1")
    display(comparison_df)
else:
    print("No tuned metrics found yet. Run the main pipeline cell first.")

Baseline best num_boost_round: 200
Baseline best CV F-beta: 0.9194


,num_boost_round,cv_precision_mean,cv_recall_mean,cv_fbeta_mean
0,100,0.877578,0.950168,0.918427
1,200,0.888994,0.943053,0.919363
2,400,0.890831,0.931204,0.913670
3,600,0.888692,0.928852,0.911377


              precision    recall  f1-score   support

           0       0.81      0.77      0.79        98
           1       0.75      0.79      0.77        86

    accuracy                           0.78       184
   macro avg       0.78      0.78      0.78       184
weighted avg       0.78      0.78      0.78       184

Baseline metrics: {'model': 'xgboost_baseline_cv', 'threshold': 0.5, 'accuracy': 0.7771739130434783, 'roc_auc': 0.8730422401518747, 'f1': 0.768361581920904, 'precision': 0.7472527472527473, 'recall': 0.7906976744186046, 'specificity': np.float64(0.7653061224489796), 'confusion_matrix': array([[75, 23],
       [18, 68]])}

Baseline vs Tuned Top-1


,model,accuracy,precision,recall,f1,roc_auc,specificity
0,baseline_cv,0.777174,0.747253,0.790698,0.768362,0.873042,0.765306
1,tuned_top1,0.777174,0.731959,0.825581,0.775956,0.877195,0.734694


In [139]:
# MLflow logging: top-3 tuned XGBoost models + metrics (with unique, consistent naming)
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import re
import mlflow
import mlflow.xgboost
import xgboost

if "cfg" not in globals():
    cfg = load_config("config.yaml")
if "xgb_results" not in globals() or "xgb_metrics" not in globals():
    raise RuntimeError("Run the XGBoost pipeline/evaluation cells first.")

def _slug(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_") or "na"

tracking_dir = Path("mlruns").resolve()
mlflow.set_tracking_uri(tracking_dir.as_uri())

data_cfg = cfg.get("data", {})
xgb_cfg = cfg.get("training", {}).get("xgboost", {})
eval_cfg = cfg.get("evaluation", {})

label_slug = _slug(data_cfg.get("label_column", "heart_disease"))
run_slug = _slug(xgb_cfg.get("run_name", "xgboost_tuned"))
experiment_name = eval_cfg.get(
    "mlflow_experiment_name",
    f"iomt_ml.{label_slug}.{run_slug}.tuned"
)
mlflow.set_experiment(experiment_name)

metric_name = xgb_cfg.get("metric_name", "validation-fbeta12")
model_filename = eval_cfg.get("xgboost_model_filename", "model.ubj")
top_n = int(eval_cfg.get("xgboost_top_n", 3))
session_uid = f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid4().hex[:8]}"

top_results = sorted(
    [r for r in xgb_results if not r.error],
    key=lambda r: r.metrics.get(metric_name, float("-inf")),
    reverse=True,
)[:top_n]

for i, r in enumerate(top_results, start=1):
    rank_uid = f"r{i:02d}_{session_uid}"
    run_name = f"{run_slug}.{rank_uid}"
    model_artifact_name = f"xgboost_model_{rank_uid}"

    with mlflow.start_run(run_name=run_name, nested=False):
        mlflow.set_tag("notebook", "xgboost_trainer")
        mlflow.set_tag("experiment_name", experiment_name)
        mlflow.set_tag("session_uid", session_uid)
        mlflow.set_tag("rank", i)
        mlflow.set_tag("model_family", "xgboost")
        mlflow.set_tag("model_artifact_name", model_artifact_name)

        # Tune-level params/metric
        mlflow.log_params({k: str(v) for k, v in r.config.items()})
        mlflow.log_metric(metric_name, float(r.metrics.get(metric_name, float("nan"))))

        # Evaluation metrics from xgb_metrics list (same ranking order)
        eval_row = xgb_metrics[i - 1] if len(xgb_metrics) >= i else {}

        for m in [
            "accuracy",
            "precision_label_1",
            "recall_label_1",
            "f1_label_1",
            "roc_auc",
            "specificity",
            "tune_metric_value",
        ]:
            if m in eval_row:
                mlflow.log_metric(m, float(eval_row[m]))

        cm = eval_row.get("confusion_matrix")
        if cm is not None:
            cm_arr = np.asarray(cm)
            if cm_arr.shape == (2, 2):
                mlflow.log_metric("tn", float(cm_arr[0, 0]))
                mlflow.log_metric("fp", float(cm_arr[0, 1]))
                mlflow.log_metric("fn", float(cm_arr[1, 0]))
                mlflow.log_metric("tp", float(cm_arr[1, 1]))

        # Load and log booster artifact from checkpoint
        ckpt = r.checkpoint
        if ckpt is None:
            raise ValueError(f"Top-{i} trial has no checkpoint to log.")

        with ckpt.as_directory() as checkpoint_dir:
            booster = xgboost.Booster()
            booster.load_model(str(Path(checkpoint_dir) / model_filename))
            mlflow.xgboost.log_model(booster, name=model_artifact_name)

        print(f"Logged to MLflow: run={run_name}, model={model_artifact_name}")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow experiment: {experiment_name}")
print(f"MLflow session UID: {session_uid}")

2026/03/31 06:06:53 INFO mlflow.tracking.fluent: Experiment with name 'iomt_ml.heartdisease.heartdisease_prediction_xgboost.tuned' does not exist. Creating a new experiment.


Logged to MLflow: run=heartdisease_prediction_xgboost.r01_20260330T220653Z_bcaf1110, model=xgboost_model_r01_20260330T220653Z_bcaf1110
Logged to MLflow: run=heartdisease_prediction_xgboost.r02_20260330T220653Z_bcaf1110, model=xgboost_model_r02_20260330T220653Z_bcaf1110
Logged to MLflow: run=heartdisease_prediction_xgboost.r03_20260330T220653Z_bcaf1110, model=xgboost_model_r03_20260330T220653Z_bcaf1110
MLflow tracking URI: file:///C:/Users/BIZZ/Documents/Vscode/python/IoMT%20ML/mlruns
MLflow experiment: iomt_ml.heartdisease.heartdisease_prediction_xgboost.tuned
MLflow session UID: 20260330T220653Z_bcaf1110
